Análise da Dados 

In [9]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import ast

In [10]:
negociaçoes = r"C:\Users\Orçamento\OneDrive - GRUPO RETEC\02. Engenharia\Dep. Orçamentos\POWERBI\AUTOMACAO RD\data\negociacoes_2025.xlsx"
df = pd.read_excel(negociaçoes)

In [11]:
print(df.head)

<bound method NDFrame.head of                            _id                        id  \
0     6824ce26f8714000245cb496  6824ce26f8714000245cb496   
1     6824b9d01ed13a001b86a4b3  6824b9d01ed13a001b86a4b3   
2     68249a57b27c77001f3cb5c4  68249a57b27c77001f3cb5c4   
3     68249a4a1407630018cfea3e  68249a4a1407630018cfea3e   
4     68249869ecf60c001fb4c957  68249869ecf60c001fb4c957   
...                        ...                       ...   
2903  677685aac7ad66001f69dbc5  677685aac7ad66001f69dbc5   
2904  6776855e2d52460014191210  6776855e2d52460014191210   
2905  6776855279d2c00017fdea61  6776855279d2c00017fdea61   
2906  677679be82e94600174715a2  677679be82e94600174715a2   
2907  677676ac2d5246001818fd9d  677676ac2d5246001818fd9d   

                                                   name  amount_montly  \
0               NCP- PROLIMA OBRA: CAMARA DOS DEPUTADOS            0.0   
1               NCP 12356 - DOENÇAS RARAS BSB - NOVACAP            0.0   
2       NCP-12354 MONUMENTA

In [12]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2908 entries, 0 to 2907
Data columns (total 88 columns):
 #   Column                                           Non-Null Count  Dtype  
---  ------                                           --------------  -----  
 0   _id                                              2908 non-null   object 
 1   id                                               2908 non-null   object 
 2   name                                             2908 non-null   object 
 3   amount_montly                                    2908 non-null   float64
 4   amount_unique                                    2908 non-null   float64
 5   amount_total                                     2908 non-null   float64
 6   prediction_date                                  3 non-null      object 
 7   markup                                           2908 non-null   object 
 8   last_activity_at                                 1 non-null      object 
 9   interactions                  

In [13]:
df.describe()

,amount_montly,amount_unique,amount_total,interactions,rating,hold,win,stop_time_limit.expired,stop_time_limit.expired_days,organization.address_latitude,organization.address_longitude,NPS - Como o Cliente avalia o atendimento,Observações
count,2908.000000,2.908000e+03,2.908000e+03,2908.000000,2908.000000,0.0,1316.000000,302.000000,302.000000,1.000000,1.000000,0.0,0.0
mean,69.111372,3.725970e+04,3.732881e+04,1.348349,2.282325,NaN,0.910334,0.811258,31.175497,-16.708663,-49.235882,NaN,NaN
std,3726.891215,4.751229e+05,4.751321e+05,2.620217,1.831499,NaN,0.285811,0.391953,30.720782,NaN,NaN,NaN,NaN
min,0.000000,0.000000e+00,0.000000e+00,0.000000,1.000000,NaN,0.000000,0.000000,0.000000,-16.708663,-49.235882,NaN,NaN
25%,0.000000,0.000000e+00,0.000000e+00,0.000000,1.000000,NaN,1.000000,1.000000,3.000000,-16.708663,-49.235882,NaN,NaN
50%,0.000000,1.369900e+02,1.380450e+02,1.000000,1.000000,NaN,1.000000,1.000000,20.000000,-16.708663,-49.235882,NaN,NaN
75%,0.000000,2.043910e+03,2.056970e+03,1.000000,5.000000,NaN,1.000000,1.000000,54.000000,-16.708663,-49.235882,NaN,NaN
max,200975.870000,1.339870e+07,1.339870e+07,27.000000,5.000000,NaN,1.000000,1.000000,126.000000,-16.708663,-49.235882,NaN,NaN


In [14]:
df.columns

Index(['_id', 'id', 'name', 'amount_montly', 'amount_unique', 'amount_total',
       'prediction_date', 'markup', 'last_activity_at', 'interactions',
       'markup_last_activities', 'created_at', 'updated_at', 'rating',
       'markup_created', 'last_activity_content', 'user_changed', 'hold',
       'win', 'closed_at', 'contacts', 'deal_custom_fields', 'deal_products',
       'stop_time_limit.expiration_date_time', 'stop_time_limit.expired',
       'stop_time_limit.expired_days', 'organization._id', 'organization.id',
       'organization.name', 'organization.address',
       'organization.address_latitude', 'organization.address_longitude',
       'organization.user._id', 'organization.user.id',
       'organization.user.name', 'organization.user.email',
       'organization.organization_segments', 'user._id', 'user.id',
       'user.name', 'user.nickname', 'user.email', 'deal_stage._id',
       'deal_stage.id', 'deal_stage.name', 'deal_stage.nickname',
       'deal_stage.created_at'

In [15]:


# 2) Garanta que a coluna seja lista de dicts
def parse_custom_fields(s):
    if isinstance(s, str):
        try:
            return ast.literal_eval(s)
        except (ValueError, SyntaxError):
            return []
    elif isinstance(s, list):
        return s
    else:
        return []
df.loc[:, "deal_custom_fields"] = df["deal_custom_fields"].apply(parse_custom_fields)

# 3) Função que retorna um dict {label: value} para cada linha
def expandir_campos(campos):
    resultado = {}
    for campo in campos:
        cf = campo.get("custom_field", {})
        label = cf.get("label")
        if label:
            resultado[label] = campo.get("value")
    return resultado

# 4) Aplique a expansão e transforme em DataFrame
df_custom = df["deal_custom_fields"] \
    .apply(expandir_campos) \
    .apply(pd.Series)

# 5) Una ao DataFrame original
df_expanded = pd.concat([df, df_custom], axis=1)

# Agora cada label virou uma coluna, ex:
print(df_expanded.columns)       # verá colunas como 'Fator', 'Proposta Nº', 'Unidade de Negócio', …
print(df_expanded[["id", "user.name", "Fator", "Proposta Nº", "Unidade de Negócio"]].head())


Index(['_id', 'id', 'name', 'amount_montly', 'amount_unique', 'amount_total',
       'prediction_date', 'markup', 'last_activity_at', 'interactions',
       ...
       'Produtos (Representação)', 'ArControl',
       'Previsão do Resultado da Concorrência', 'Fábrica (Representação)',
       'Data Limite para Recebimento do Produto na Obra', 'Chance de Ganho',
       'Previsão de Fechamento da Negociação', 'Orçamentista',
       'NPS - Como o Cliente avalia o atendimento', 'Observações'],
      dtype='object', length=107)
                         id         user.name Fator Fator Proposta Nº  \
0  6824ce26f8714000245cb496  Wellisson Chaves    01    01         NaN   
1  6824b9d01ed13a001b86a4b3       Luan Araújo   NaN  None       12356   
2  68249a57b27c77001f3cb5c4  Wellisson Chaves    01    01       12354   
3  68249a4a1407630018cfea3e  Wellisson Chaves    01    01       12354   
4  68249869ecf60c001fb4c957  Wellisson Chaves    01    01       14060   

  Proposta Nº Unidade de Negócio Un

In [16]:
def parse_custom_fields(s):
    if isinstance(s, str):
        try:
            return ast.literal_eval(s)
        except (ValueError, SyntaxError):
            return []
    elif isinstance(s, list):
        return s
    else:
        return []

df.loc[:, "deal_custom_fields"] = df["deal_custom_fields"].apply(parse_custom_fields)

# 3) Função que extrai o valor de “Fator”
def extrair_fator(campos):
    for campo in campos:
        cf = campo.get("custom_field", {})
        if cf.get("label", "").strip().lower() == "fator":
            return campo.get("value")
    return None

# 4) Aplique linha a linha e crie a coluna 'fator'
df.loc[:, "fator"] = df["deal_custom_fields"].apply(extrair_fator)

# 5) (Opcional) Converter string com vírgula para float
df.loc[:, "fator"] = (
    df["fator"]
    .astype(str)
    .str.replace(",", ".", regex=False)
    .astype(float, errors="ignore")
)

# 6) Filtrar só Bruno Crispim e Gabriel Bento e mostrar
vendedores = ["Bruno Crispim", "Gabriel  Bento"]
resultado = df[df["user.name"].isin(vendedores)][["id", "user.name", "fator"]]
print(resultado.head(20))

                           id       user.name   fator
94   681cb3ab978e2f00275fb27d   Bruno Crispim    0.85
95   681cae7e320c23001baa3e1d   Bruno Crispim    0.62
96   681cad029c91db001413c94a   Bruno Crispim       1
134  6818bf0a03c9f10014ecc56b   Bruno Crispim       1
135  6818bea73d2c6e0027bdcc88   Bruno Crispim       1
136  6818baf8ef82eb002c8ee57c   Bruno Crispim       1
147  6817d7e69bf42000143ae859  Gabriel  Bento    0.73
148  6817cc312f87ea002799dc84  Gabriel  Bento     0.6
149  6817cae7ae9fbd00148d8060  Gabriel  Bento       1
150  6817c9e49bf42000273ae212  Gabriel  Bento     0.8
151  6817c8d6473aea00174b1f1f  Gabriel  Bento       1
159  681510ed06910f0014cb7098   Bruno Crispim     0.7
175  6812921d25f0cc0025342649  Gabriel  Bento     0.6
176  6812920cbded940014949eaa  Gabriel  Bento     0.6
215  68113ad73bd702001493746d   Bruno Crispim     0.8
223  6811391ea12f2600210eda82   Bruno Crispim     0.8
268  6810e0be6f2584001be8d1b2  Gabriel  Bento    0.54
269  6810e0adaa4450001448da7

Análise, criando o dataframe com as colunas que queremos analisar

In [17]:
colunas_corrigidas = [
    "id", "name", "amount_total", "amount_unique", "markup",
    "created_at", "closed_at", "last_activity_at",
    "interactions", "win", "deal_stage.name", "user.id", "user.name",
    "deal_lost_reason.name",'fator'
]

df_analise = df[colunas_corrigidas].copy()

Transformando a coluna de Datas em tipo data

In [18]:
df_analise['created_at'] = pd.to_datetime(df_analise['created_at'], errors = 'coerce')
df_analise['closed_at'] = pd.to_datetime(df_analise['closed_at'], errors = 'coerce')
df_analise["last_activity_at"] = pd.to_datetime(df_analise["last_activity_at"], errors="coerce")

In [19]:
df_analise.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2908 entries, 0 to 2907
Data columns (total 15 columns):
 #   Column                 Non-Null Count  Dtype                    
---  ------                 --------------  -----                    
 0   id                     2908 non-null   object                   
 1   name                   2908 non-null   object                   
 2   amount_total           2908 non-null   float64                  
 3   amount_unique          2908 non-null   float64                  
 4   markup                 2908 non-null   object                   
 5   created_at             2908 non-null   datetime64[ns, UTC-03:00]
 6   closed_at              1316 non-null   datetime64[ns, UTC-03:00]
 7   last_activity_at       1 non-null      datetime64[ns, UTC-03:00]
 8   interactions           2908 non-null   int64                    
 9   win                    1316 non-null   float64                  
 10  deal_stage.name        2908 non-null   object   

In [20]:
df_analise.head()

,id,name,amount_total,amount_unique,markup,created_at,closed_at,last_activity_at,interactions,win,deal_stage.name,user.id,user.name,deal_lost_reason.name,fator
0,6824ce26f8714000245cb496,NCP- PROLIMA OBRA: CAMARA DOS DEPUTADOS,0.00,0.00,future,2025-05-14 14:08:54.471000-03:00,NaT,NaT,0,NaN,Orç. Pendente (Venda),651af77665445f0022476065,Wellisson Chaves,NaN,01
1,6824b9d01ed13a001b86a4b3,NCP 12356 - DOENÇAS RARAS BSB - NOVACAP,0.00,0.00,future,2025-05-14 12:42:08.323000-03:00,NaT,NaT,0,NaN,Consulta,651af6998076130019a8dda8,Luan Araújo,NaN,None
2,68249a57b27c77001f3cb5c4,NCP-12354 MONUMENTAL PROJETOS OBRA: ESCRITORIOI,0.00,0.00,future,2025-05-14 10:27:51.495000-03:00,NaT,NaT,0,NaN,Venda Ganha,651af77665445f0022476065,Wellisson Chaves,NaN,01
3,68249a4a1407630018cfea3e,NCP-12354 MONUMENTAL PROJETOS OBRA: ESCRITORIOI,172.47,172.47,future,2025-05-14 10:27:39.023000-03:00,2025-05-14 10:27:51.371000-03:00,NaT,1,1.0,Venda Ganha,651af77665445f0022476065,Wellisson Chaves,NaN,01
4,68249869ecf60c001fb4c957,NCP-14060 PRO-HAB OBRA: BRB 53,0.00,0.00,future,2025-05-14 10:19:37.740000-03:00,NaT,NaT,0,NaN,Venda Ganha,651af77665445f0022476065,Wellisson Chaves,NaN,01


In [21]:
# Converter campo win em booleano
df_analise["win"] = df_analise["win"].map({"true": True, "false": False})

# Recriar colunas auxiliares
df_analise["duracao_venda_dias"] = (df_analise["closed_at"] - df_analise["created_at"]).dt.days
df_analise["mes_criacao"] = df_analise["created_at"].dt.to_period("M")
#df_analise["status_final"] = df_analise["win"].map({True: "Ganho", False: "Perdido"})
#df_analise.loc[df_analise["closed_at"].isna(), "status_final"] = "Em Aberto"

C:\Users\Orçamento\AppData\Local\Temp\ipykernel_16444\2614631020.py:6: UserWarning: Converting to PeriodArray/Index representation will drop timezone information.
  df_analise["mes_criacao"] = df_analise["created_at"].dt.to_period("M")


In [22]:
df_analise[::-10]

,id,name,amount_total,amount_unique,markup,created_at,closed_at,last_activity_at,interactions,win,deal_stage.name,user.id,user.name,deal_lost_reason.name,fator,duracao_venda_dias,mes_criacao
2907,677676ac2d5246001818fd9d,NCP 10317 - CASA VARANDA - ISRAEL GRADO ENGENH...,308.32,308.32,future,2025-01-02 08:21:16.306000-03:00,2024-12-31 01:00:00-03:00,NaT,1,NaN,Venda Ganha,651af6998076130019a8dda8,Luan Araújo,NaN,1,-3.0,2025-01
2897,6776c1f3d65038001edb2955,8098 - GEOLAB - GLEICIANE,28092.88,28092.88,future,2025-01-02 13:42:27.410000-03:00,NaT,NaT,1,NaN,Negociação,651af40601a5360011f74f6c,Rutemar Júnior,NaN,1,NaN,2025-01
2887,6777c299d650380018db8bed,NCP 10326 - SARAH LAGO NORTE - JICLIMAR REFRIG...,0.00,0.00,future,2025-01-03 07:57:29.274000-03:00,NaT,NaT,0,NaN,Venda Ganha,651af6998076130019a8dda8,Luan Araújo,NaN,1,NaN,2025-01
2877,6777ea816fbbb3001b6e5a94,NCP 10315 - FUNCEF - ALTRI CONSTRUTORA,8648.41,8648.41,future,2025-01-03 10:47:46.014000-03:00,2025-02-03 15:21:02.397000-03:00,NaT,11,NaN,Venda Perdida,651af6998076130019a8dda8,Luan Araújo,Relacionamento (Confiança e Parceria),1,31.0,2025-01
2867,6778280080810e001f34511d,NCP 10313 - CIVIL EMG - ARAUJO ABREU ENGENHARIA,1007.19,1007.19,future,2025-01-03 15:10:08.890000-03:00,2025-01-03 15:59:55.427000-03:00,NaT,1,NaN,Venda Ganha,651af6998076130019a8dda8,Luan Araújo,NaN,1,0.0,2025-01
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
47,681d39d5fc6f4e001b916d8e,12286 - CAIXA ECONOMICA - MARUS,0.00,0.00,future,2025-05-08 20:10:13.146000-03:00,NaT,NaT,0,NaN,Venda Ganha,65f878886cab6d0015e4690f,Iago Rangel,NaN,1,NaN,2025-05
37,681e643ef398f800164d8a16,NCP-12257 ALMEIDA GOMES OBRA: BANCO DO BRASIL,7942.80,7942.80,future,2025-05-09 17:23:26.462000-03:00,NaT,NaT,0,NaN,Negociação,651af77665445f0022476065,Wellisson Chaves,NaN,01,NaN,2025-05
27,682342d5ce93280020b7a00d,NCP - 12311 LM ENGENHARIA DE REFRIGERACAO LTDA,1536.36,1536.36,future,2025-05-13 10:02:13.782000-03:00,2025-05-13 10:03:05.500000-03:00,NaT,1,NaN,Venda Ganha,655ca6a9d691d60021c81701,Marlon Souza,NaN,1,0.0,2025-05
17,6823a485ed265600199b0623,NCP - 12350 VITURINO XIMENS NETO - ME,0.00,0.00,future,2025-05-13 16:59:01.240000-03:00,NaT,NaT,0,NaN,Venda Ganha,655ca6a9d691d60021c81701,Marlon Souza,NaN,1,NaN,2025-05


Estatísticas principais

In [23]:
df_analise['fator'] = pd.to_numeric(df_analise['fator'], errors='coerce')

In [24]:
resumo = {
    "Total de negociações": len(df_analise),
    "Contagem de Vendas Ganhas" : df_analise[df_analise["deal_stage.name"] == "Venda Ganha"].shape[0],
    "Contagem de Vendas Perdidas" : df_analise[df_analise["deal_stage.name"] == "Venda Perdida"].shape[0],
    "Contagem de Vendas Canceladas" : df_analise[df_analise["deal_stage.name"] == "Venda Cancelada"].shape[0],
    "Contagem de Vendas em aberto" : df_analise[df_analise["deal_stage.name"] == "Negociação"].shape[0],
    "Valor Total Ganhas": float(df_analise[df_analise["deal_stage.name"] == "Venda Ganha"]["amount_total"].sum()),
    "Valor Total Perdidas": float(df_analise[df_analise["deal_stage.name"] == "Venda Perdida"]["amount_total"].sum()),
    "Valor Total Canceladas": float(df_analise[df_analise["deal_stage.name"] == "Venda Cancelada"]["amount_total"].sum()),
    "Valor Total Não vendas": float(df_analise[df_analise["deal_stage.name"].isin(["Venda Perdida", "Venda Cancelada"])]["amount_total"].sum()),
    "Valor Total em Aberto": float(df_analise[df_analise["deal_stage.name"] == "Negociação"]["amount_total"].sum()),
    "Ticket Médio Ganho": float(df_analise[df_analise["deal_stage.name"] == "Venda Ganha"]["amount_total"].mean()),
    "Taxa de Desconto" : (float(df_analise['fator'].mean())),
    "Duração média (dias)": float(df_analise["duracao_venda_dias"].mean())
}

In [25]:
resumo

{'Total de negociações': 2908,
 'Contagem de Vendas Ganhas': 2422,
 'Contagem de Vendas Perdidas': 90,
 'Contagem de Vendas Canceladas': 27,
 'Contagem de Vendas em aberto': 128,
 'Valor Total Ganhas': 6297758.15,
 'Valor Total Perdidas': 3061888.3100000005,
 'Valor Total Canceladas': 1106928.4,
 'Valor Total Não vendas': 4168816.71,
 'Valor Total em Aberto': 10248546.3,
 'Ticket Médio Ganho': 2600.2304500412883,
 'Taxa de Desconto': 11.906318005540165,
 'Duração média (dias)': 3.519756838905775}

Dados apenas dos vendedores da Representação (Bruno e Bento)

In [26]:
df_rep = df_analise[df_analise['user.name'].isin(['Bruno Crispim', 'Gabriel  Bento'])]
df_rep

,id,name,amount_total,amount_unique,markup,created_at,closed_at,last_activity_at,interactions,win,deal_stage.name,user.id,user.name,deal_lost_reason.name,fator,duracao_venda_dias,mes_criacao
94,681cb3ab978e2f00275fb27d,NCT 13989-25 - TECNA CONSTRUTORA - RESIDENCIAL...,0.00,0.0,future,2025-05-08 10:37:47.556000-03:00,NaT,NaT,0,NaN,Consulta Enviada,6557ecc0295062000f0ac40a,Bruno Crispim,NaN,0.85,NaN,2025-05
95,681cae7e320c23001baa3e1d,NCD 25050234-25 - FOX ENGENHARIA - BRB - 06.05...,200975.87,0.0,future,2025-05-08 10:15:42.148000-03:00,NaT,NaT,0,NaN,Consulta Enviada,6557ecc0295062000f0ac40a,Bruno Crispim,NaN,0.62,NaN,2025-05
96,681cad029c91db001413c94a,NCS 234974-25 - FOX ENGENHARIA - BRB - 06.05.2...,8962.60,8962.6,future,2025-05-08 10:09:22.366000-03:00,NaT,NaT,0,NaN,Consulta Enviada,6557ecc0295062000f0ac40a,Bruno Crispim,NaN,1.00,NaN,2025-05
134,6818bf0a03c9f10014ecc56b,NOP 109282-25 - UNIAO QUIMICA - VIRICAS I - 22...,7600.00,7600.0,future,2025-05-05 10:37:14.945000-03:00,NaT,NaT,0,NaN,Consulta Enviada,6557ecc0295062000f0ac40a,Bruno Crispim,NaN,1.00,NaN,2025-05
135,6818bea73d2c6e0027bdcc88,NCT-X 13965-25 - UNIAO QUIMICA - VIRICAS I - 2...,29000.00,29000.0,future,2025-05-05 10:35:35.514000-03:00,NaT,NaT,0,NaN,Consulta Enviada,6557ecc0295062000f0ac40a,Bruno Crispim,NaN,1.00,NaN,2025-05
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2843,677c3b699321820027e7d48c,NCI 3369-24 - CONNECTOR - MULTI CONSTRUTORA - ...,61000.00,61000.0,future,2025-01-06 17:22:01.459000-03:00,2025-01-30 11:45:59.249000-03:00,NaT,4,NaN,Venda Perdida,6557ecc0295062000f0ac40a,Bruno Crispim,Preço (perca para produto inferior),0.65,23.0,2025-01
2844,677c394b7aa09300141393a6,NCD 0050-24 - SOUSAR CLIMAENGENHARIA - SQS 303...,78958.70,78958.7,future,2025-01-06 17:12:59.268000-03:00,NaT,NaT,3,NaN,Orç. Enviar (Venda),6557ecc0295062000f0ac40a,Bruno Crispim,NaN,0.45,NaN,2025-01
2845,677c3882a38d3a0018b260fc,NCD 25010141-24 - NORTHEC - SESI - 03.01.2025 ...,0.00,0.0,future,2025-01-06 17:09:38.197000-03:00,NaT,NaT,0,NaN,Consulta Enviada,6557ecc0295062000f0ac40a,Bruno Crispim,NaN,0.45,NaN,2025-01
2847,677c368ac1cc7300169f18eb,NCD 0049 -REV2- RENOVAR ENGENHARIA - CAIXA ECO...,0.00,0.0,future,2025-01-06 17:01:14.939000-03:00,NaT,NaT,0,NaN,Venda Ganha,6557ecc0295062000f0ac40a,Bruno Crispim,NaN,0.39,NaN,2025-01


In [27]:
total_orc = len(df_rep["deal_stage.name"] == "Venda Ganha") + len(df_rep["deal_stage.name"] == "Venda Perdida") + len(df_rep["deal_stage.name"] == "Venda Cancelada")
taxa_conversão_rep = df_rep[df_rep["deal_stage.name"] == "Venda Ganha"].shape[0]/total_orc
df_rep['fator'] = df_rep['fator'].astype(float)

C:\Users\Orçamento\AppData\Local\Temp\ipykernel_16444\2486677785.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_rep['fator'] = df_rep['fator'].astype(float)


In [28]:
resumo_rep = {
    "Total de negociações": len(df_rep),
    "Contagem de Vendas Ganhas" : df_rep[df_rep["deal_stage.name"] == "Venda Ganha"].shape[0],
    "Contagem de Vendas Perdidas" : df_rep[df_rep["deal_stage.name"] == "Venda Perdida"].shape[0],
    "Contagem de Vendas Canceladas" : df_rep[df_rep["deal_stage.name"] == "Venda Cancelada"].shape[0],
    "Contagem de Vendas em aberto" : df_rep[df_rep["deal_stage.name"] == "Negociação"].shape[0],
    "Valor Total Ganhas": float(df_rep[df_rep["deal_stage.name"] == "Venda Ganha"]["amount_total"].sum()),
    "Valor Total Perdidas": float(df_rep[df_rep["deal_stage.name"] == "Venda Perdida"]["amount_total"].sum()),
    "Valor Total Canceladas": float(df_rep[df_rep["deal_stage.name"] == "Venda Cancelada"]["amount_total"].sum()),
    "Valor Total Não vendas": float(df_rep[df_rep["deal_stage.name"].isin(["Venda Perdida", "Venda Cancelada"])]["amount_total"].sum()),
    "Valor Total em Aberto": float(df_rep[df_rep["deal_stage.name"] == "Negociação"]["amount_total"].sum()),
    "Ticket Médio Ganho": float(df_rep[df_rep["deal_stage.name"] == "Venda Ganha"]["amount_total"].mean()),
    "Taxa de Conversão ": float(taxa_conversão_rep),
    "Taxa de Desconto" : (1 - float(df_rep['fator'].mean())),
    "Duração média (dias)": float(df_rep["duracao_venda_dias"].mean())
}

In [29]:
resumo_rep

{'Total de negociações': 263,
 'Contagem de Vendas Ganhas': 100,
 'Contagem de Vendas Perdidas': 19,
 'Contagem de Vendas Canceladas': 6,
 'Contagem de Vendas em aberto': 32,
 'Valor Total Ganhas': 3160263.28,
 'Valor Total Perdidas': 2088878.7899999998,
 'Valor Total Canceladas': 574132.76,
 'Valor Total Não vendas': 2663011.5500000003,
 'Valor Total em Aberto': 8022858.470000001,
 'Ticket Médio Ganho': 31602.6328,
 'Taxa de Conversão ': 0.1267427122940431,
 'Taxa de Desconto': 0.15609733840304174,
 'Duração média (dias)': 7.739726027397261}